# NB_02 — Silver Transformation

Transforms raw Bronze data into clean, typed, and enriched Silver tables.

Silver is the single source of truth. Any analyst or engineer needing reliable data starts here.
The rules are simple: fix types, handle nulls explicitly, deduplicate keys, correct known source errors.
No business logic, no aggregations — those belong in Gold.

What this notebook does:
- Casts all columns to correct types
- Parses dates using the exact source format (M/d/yyyy, no zero-padding)
- Rounds floating-point artifacts in Discount values (IEEE 754 fix)
- Fixes known source typos in Categories and Suppliers
- Resolves the DivisionID=2 primary key violation by quarantining the duplicate
- Derives DivisionID from Country using a geographic reference table
- Adds geographic enrichment: Continent, SubRegion, ISO2 country code
- Injects Unknown Member records (ID = -1) to preserve financial integrity
- Computes derived fields: margin bands, price tiers, fiscal periods
- Sends invalid records to bronze_quarantine instead of silently dropping them

What this notebook does NOT do:
- Join facts and dimensions (Gold)
- Compute aggregations or KPIs (Gold)
- Apply business rules like seasonality or budget allocation (Gold)


In [ ]:
# In order to be able to make data enriched downstream - Data curated for geo reference. 
# Type here in the cell editor to add code!
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window
import re, pandas as pd

def clean_cols(df):
    # Replace spaces and special chars in column names for Delta compatibility
    new_cols = [re.sub(r'[ ,;{}()\n\t=]', '_', c).strip('_') for c in df.columns]
    for old, new in zip(df.columns, new_cols):
        if old != new:
            df = df.withColumnRenamed(old, new)
    return df

# Geographic reference: Country -> correct DivisionID, Continent, SubRegion, ISO2
# This resolves the source issue where DivisionID=2 was assigned to both
# North America and Central America (primary key violation in Divisions.csv)
geo_ref = [
    ("AUSTRIA",1,"Europe","Western Europe","AT"),
    ("BELGIUM",1,"Europe","Western Europe","BE"),
    ("DENMARK",1,"Europe","Northern Europe","DK"),
    ("FINLAND",1,"Europe","Northern Europe","FI"),
    ("FRANCE",1,"Europe","Western Europe","FR"),
    ("GERMANY",1,"Europe","Western Europe","DE"),
    ("IRELAND",1,"Europe","Western Europe","IE"),
    ("ITALY",1,"Europe","Southern Europe","IT"),
    ("POLAND",1,"Europe","Eastern Europe","PL"),
    ("PORTUGAL",1,"Europe","Southern Europe","PT"),
    ("SPAIN",1,"Europe","Southern Europe","ES"),
    ("SWITZERLAND",1,"Europe","Western Europe","CH"),
    ("UK",1,"Europe","Northern Europe","GB"),
    ("USA",2,"Americas","North America","US"),
    ("CANADA",2,"Americas","North America","CA"),
    ("NORWAY",3,"Europe","Northern Europe","NO"),
    ("SWEDEN",3,"Europe","Northern Europe","SE"),
    ("ARGENTINA",4,"Americas","South America","AR"),
    ("BRAZIL",4,"Americas","South America","BR"),
    ("VENEZUELA",4,"Americas","South America","VE"),
    ("MEXICO",6,"Americas","Central America","MX"),  # remapped from duplicate DivisionID=2
    ("JAPAN",5,"Asia","East Asia","JP"),
    ("SINGAPORE",5,"Asia","Southeast Asia","SG"),
    ("AUSTRALIA",7,"Oceania","Australasia","AU"),
    ("NETHERLANDS",1,"Europe","Western Europe","NL"),
]

schema_geo = StructType([
    StructField("Country_Upper",StringType()),
    StructField("DivisionID",IntegerType()),
    StructField("Continent",StringType()),
    StructField("SubRegion",StringType()),
    StructField("ISO2",StringType()),
])
df_geo = spark.createDataFrame(geo_ref, schema_geo)

print(f"Setup complete | geographic reference: {len(geo_ref)} countries mapped")


In [ ]:
# In order to keep governance over the data enviroment decided to creat a layer for a Quarantine framework
# Invalid records that cannot be automatically fixed go to bronze_quarantine
# instead of being silently dropped or corrected without a trace.
# The table accumulates issues across runs for governance and audit purposes.

def quarantine(df, check_name, source_table, reason):
    df_bad = (df
        .withColumn("_quarantine_reason", lit(reason))
        .withColumn("_quarantine_check",  lit(check_name))
        .withColumn("_quarantine_source", lit(source_table))
        .withColumn("_quarantined_at",    current_timestamp())
    )
    df_bad.write.format("delta").mode("append").option("mergeSchema", True).saveAsTable("bronze_quarantine")
    print(f"  QUARANTINE [{check_name}]: {df_bad.count()} records sent to bronze_quarantine")

print("Quarantine framework ready")

In [ ]:
# silver_divisions
# Critical source issue: DivisionID=2 appears twice with different names
# (North America and Central America) — this is a primary key violation.
# Action: keep the first valid record, quarantine the duplicate,
# then inject Central America with a new unique ID (6).

from datetime import datetime

df_raw = spark.table("bronze_divisions")
df_div = df_raw.withColumn("DivisionID", col("DivisionID").cast("integer"))

w = Window.partitionBy("DivisionID").orderBy(monotonically_increasing_id())
df_div = df_div.withColumn("_rn", row_number().over(w))

df_dupes = df_div.filter(col("_rn") > 1).drop("_rn")
if df_dupes.count() > 0:
    quarantine(df_dupes, "duplicate_primary_key", "bronze_divisions",
        "DivisionID=2 is duplicated in source — Central America conflicts with North America. Source correction required.")

df_clean = df_div.filter(col("_rn") == 1).drop("_rn").withColumn("_silver_at", current_timestamp())

now = datetime.now()
central_america = spark.createDataFrame(
    [(6, "Central America", now, now)],
    ["DivisionID", "DivisionName", "_ingested_at", "_silver_at"]
)

df_div_final = df_clean.union(central_america)
df_div_final.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_divisions")
print("silver_divisions:")
df_div_final.show()


In [ ]:
# silver_shippers
# Source issue: ShipperIDs 4 and 5 appear in Orders and Shipments but have no record in Shippers.csv
# Injecting Unknown Member records preserves referential integrity in Gold.
# Without this, ~20% of orders would lose their shipper dimension and financial values would be orphaned.

from datetime import datetime

df = spark.table("bronze_shippers")
df_clean = (df
    .withColumn("ShipperID",   col("ShipperID").cast("integer"))
    .withColumn("CompanyName", trim(col("CompanyName")))
    .withColumn("_silver_at",  current_timestamp())
)

now = datetime.now()
unknowns = spark.createDataFrame([
    (4, "Unknown Shipper 4", now, now),
    (5, "Unknown Shipper 5", now, now),
], ["ShipperID", "CompanyName", "_ingested_at", "_silver_at"])

df_final = df_clean.union(unknowns)
df_final.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_shippers")
print("silver_shippers:")
df_final.show()

In [ ]:
# silver_categories
# Two source quality issues fixed:
# 1. Typo: "Weaher" in CategoryIDs 9 and 10 — corrected to "Weather"
# 2. Description "xxx" in category 9 — replaced with a meaningful value
# Note: source files are not modified per scenario rules — corrections applied in Silver only

df = spark.table("bronze_categories")
df_silver = (df
    .withColumn("CategoryID",   col("CategoryID").cast("integer"))
    .withColumn("CategoryName",
        regexp_replace(trim(col("CategoryName")), "Weaher", "Weather"))
    .withColumn("Description",
        when(trim(col("Description")) == "xxx", lit("Outdoor cold weather clothing"))
        .otherwise(trim(col("Description"))))
    .withColumn("_silver_at", current_timestamp())
)

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_categories")
print("silver_categories:")
df_silver.show(truncate=False)


In [ ]:
# silver_suppliers
# Two fixes applied:
# 1. Typo: "Dressed for Succes" corrected to "Dressed for Success"
# 2. Geographic enrichment: Continent, SubRegion, ISO2 derived from Country

df = spark.table("bronze_suppliers")
df_silver = (df
    .withColumn("SupplierID", col("SupplierID").cast("integer"))
    .withColumn("Country",    trim(upper(col("Country"))))
    .withColumn("CompanyName",
        when(col("CompanyName") == "Dressed for Succes", lit("Dressed for Success"))
        .otherwise(trim(col("CompanyName"))))
    .join(df_geo, trim(upper(col("Country"))) == col("Country_Upper"), "left")
    .drop("Country_Upper", "DivisionID")
    .withColumn("Continent",  coalesce(col("Continent"), lit("Unknown")))
    .withColumn("SubRegion",  coalesce(col("SubRegion"),  lit("Unknown")))
    .withColumn("ISO2",       coalesce(col("ISO2"),       lit("??")))
    .withColumn("_silver_at", current_timestamp())
)

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_suppliers")
print(f"silver_suppliers: {df_silver.count()} rows")
df_silver.select("SupplierID","CompanyName","Country","Continent","SubRegion").show(truncate=False)

In [ ]:
# silver_customers
# Enrichments and fixes:
# 1. DivisionID derived from Country via geographic reference — more reliable than the source field
#    (source had a PK violation that made the original DivisionID untrustworthy)
# 2. Continent, SubRegion, ISO2 added
# 3. StateProvince: NULL is correct for European and Latin American customers — made explicit as N/A
# 4. Encoding: source CSVs use Windows-1252; characters above ASCII 127 that came in corrupted are removed

df = spark.table("bronze_customers")
df_silver = (df
    .withColumn("CustomerID",   col("CustomerID").cast("integer"))
    .withColumn("Country",      trim(upper(col("Country"))))
    .withColumn("CompanyName",  trim(regexp_replace(col("CompanyName"), r'[^\x20-\x7E]', '')))
    .withColumn("ContactName",  trim(regexp_replace(col("ContactName"), r'[^\x20-\x7E]', '')))
    .withColumn("City",         trim(regexp_replace(col("City"),        r'[^\x20-\x7E]', '')))
    .withColumn("StateProvince",
        when(col("StateProvince").isNull() | (trim(col("StateProvince")) == ""), lit("N/A"))
        .otherwise(trim(col("StateProvince"))))
    .withColumn("BirthDate",    to_date(col("BirthDate"), "M/d/yyyy"))
    # Drop original DivisionID before join to avoid ambiguity
    .drop("DivisionID")
    .join(df_geo, trim(upper(col("Country"))) == col("Country_Upper"), "left")
    .drop("Country_Upper")
    .withColumn("DivisionID",   coalesce(col("DivisionID"),  lit(0)))
    .withColumn("Continent",    coalesce(col("Continent"),   lit("Unknown")))
    .withColumn("SubRegion",    coalesce(col("SubRegion"),   lit("Unknown")))
    .withColumn("ISO2",         coalesce(col("ISO2"),        lit("??")))
    .withColumn("_silver_at",   current_timestamp())
    .dropDuplicates(["CustomerID"])
    .filter(col("CustomerID").isNotNull())
)

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_customers")
print(f"silver_customers: {df_silver.count()} rows")
print("\nDivision distribution after geographic inference:")
df_silver.groupBy("DivisionID","SubRegion").count().orderBy("DivisionID").show()

In [ ]:
# silver_orders
# Fixes and enrichments:
# 1. OrderDate: format is M/d/yyyy with no zero-padding — must be specified explicitly or all dates parse as NULL
# 2. TotalOrder: 100% zero in the source — kept for audit purposes but flagged as unreliable
#    Real order totals must be computed from Order_Details (Quantity x UnitPrice x (1 - Discount))
# 3. EmployeeID NULL in 5 orders — mapped to -1 (Unknown) to preserve the financial record
# 4. Fiscal and seasonal periods derived to simplify time-based analysis in Gold

df = spark.table("bronze_orders")
df_silver = (df
    .withColumn("OrderID",    col("OrderID").cast("integer"))
    .withColumn("CustomerID", col("CustomerID").cast("integer"))
    .withColumn("ShipperID",  col("ShipperID").cast("integer"))
    .withColumn("Freight",    col("Freight").cast("double"))
    # TotalOrder is zero for all 6,571 rows in source — kept for audit, not for calculations
    .withColumn("TotalOrder_Source",   col("TotalOrder").cast("double"))
    .withColumn("TotalOrder_Reliable", lit(False))
    # NULL EmployeeID -> -1 (Unknown) to avoid orphaned sales records in Gold
    .withColumn("EmployeeID",
        when(col("EmployeeID").isNull(), lit(-1))
        .otherwise(col("EmployeeID").cast("integer")))
    # Date format: M/d/yyyy with no zero-padding (e.g. "6/29/2019", not "06/29/2019")
    .withColumn("OrderDate",  to_date(col("OrderDate"), "M/d/yyyy"))
    # Derived time attributes — avoid unnecessary joins with dim_date for simple aggregations
    .withColumn("OrderYear",    year(col("OrderDate")))
    .withColumn("OrderMonth",   month(col("OrderDate")))
    .withColumn("OrderQuarter", quarter(col("OrderDate")))
    .withColumn("OrderSeason",
        when(col("OrderMonth").isin(12,1,2),  lit("Winter"))
        .when(col("OrderMonth").isin(3,4,5),  lit("Spring"))
        .when(col("OrderMonth").isin(6,7,8),  lit("Summer"))
        .otherwise(lit("Fall")))
    # Fiscal year starts in April (last sale March 30, 2021 = FY2020)
    .withColumn("FiscalYear",
        when(col("OrderMonth") >= 4, col("OrderYear"))
        .otherwise(col("OrderYear") - 1))
    .withColumn("_silver_at", current_timestamp())
    .dropDuplicates(["OrderID"])
    .filter(col("OrderID").isNotNull())
)

null_dates = df_silver.filter(col("OrderDate").isNull()).count()
unknown_emp = df_silver.filter(col("EmployeeID") == -1).count()
print(f"silver_orders: {df_silver.count()} rows")
print(f"  Null OrderDate: {null_dates}")
print(f"  Unknown EmployeeID (-1): {unknown_emp} orders with no assigned sales rep")
print("\nOrders by year:")
df_silver.groupBy("OrderYear").count().orderBy("OrderYear").show()
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_orders")


In [ ]:
# silver_order_details
# Fixes:
# 1. Discount floating-point artifacts from source (e.g. 0.050000001 instead of 0.05)
#    Caused by IEEE 754 representation in the originating system — rounded to 2 decimal places
# 2. LineTotal computed here using the historical transaction price (UnitPrice in order_details)
#    NOT the current catalog price from Products — this distinction is critical for financial accuracy
# 3. DiscountBand added for segmentation in reports

df = spark.table("bronze_order_details")
df_silver = (df
    .withColumn("OrderID",   col("OrderID").cast("integer"))
    .withColumn("LineNo",    col("LineNo").cast("integer"))
    .withColumn("ProductID", col("ProductID").cast("integer"))
    .withColumn("Quantity",  col("Quantity").cast("integer"))
    .withColumn("UnitPrice", col("UnitPrice").cast("double"))
    # Round to 2 decimals to fix IEEE 754 artifacts: 0.150000006 -> 0.15
    .withColumn("Discount",  round(col("Discount").cast("double"), 2))
    .withColumn("LineTotal",
        round(col("Quantity").cast("double") * col("UnitPrice") * (1 - round(col("Discount").cast("double"), 2)), 2))
    .withColumn("DiscountBand",
        when(col("Discount") == 0,    lit("No Discount"))
        .when(col("Discount") < 0.10, lit("Low (1-9%)"))
        .when(col("Discount") < 0.20, lit("Medium (10-19%)"))
        .otherwise(lit("High (20%+)")))
    .withColumn("_silver_at", current_timestamp())
    .filter(col("OrderID").isNotNull())
)

print(f"silver_order_details: {df_silver.count()} rows")
print("\nDiscount values after rounding:")
df_silver.select("Discount").distinct().orderBy("Discount").show()
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_order_details")


In [ ]:
# silver_shipments
# Critical documented issue: ShipmentDate ranges from 2007 to 2012 across all 17,226 records,
# while OrderDate ranges from 2016 to 2021. Every single shipment predates its order by 5-14 years.
# This indicates the shipment data was migrated from a legacy system with a different time frame.
# Delivery SLA analysis is not reliable with this data — fact documented in _data_warning column.

df = spark.table("bronze_shipments")
df_silver = (df
    .withColumn("OrderID",      col("OrderID").cast("integer"))
    .withColumn("LineNo",       col("LineNo").cast("integer"))
    .withColumn("ShipperID",    col("ShipperID").cast("integer"))
    .withColumn("CustomerID",   col("CustomerID").cast("integer"))
    .withColumn("ProductID",    col("ProductID").cast("integer"))
    .withColumn("EmployeeID",   col("EmployeeID").cast("integer"))
    # Same date format as Orders: M/d/yyyy without zero-padding
    .withColumn("ShipmentDate", to_date(col("ShipmentDate"), "M/d/yyyy"))
    .withColumn("_data_warning",
        lit("ShipmentDate (2007-2012) predates OrderDate (2016-2021) for all records. Data from legacy system. Delivery SLA is not reliable."))
    .withColumn("_silver_at",   current_timestamp())
    .filter(col("OrderID").isNotNull())
)

date_range = df_silver.agg(min("ShipmentDate").alias("min"), max("ShipmentDate").alias("max")).collect()[0]
print(f"silver_shipments: {df_silver.count()} rows")
print(f"ShipmentDate range: {date_range['min']} to {date_range['max']}")
print("WARNING: all shipment dates predate all order dates. Known legacy data issue.")
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_shipments")


In [ ]:
# silver_products
# Enrichments added on top of type casting:
# 1. MarginAbs and MarginPct: computed from current catalog UnitCost and UnitPrice
# 2. MarginBand: Low / Medium / High segmentation for report filtering
# 3. PriceTier: Budget / Mid-Range / Premium / Luxury for product positioning analysis
# 4. InventoryStatus: current stock health indicator

df = spark.table("bronze_products")
df_silver = (df
    .withColumn("ProductID",    col("ProductID").cast("integer"))
    .withColumn("SupplierID",   col("SupplierID").cast("integer"))
    .withColumn("CategoryID",   col("CategoryID").cast("integer"))
    .withColumn("UnitCost",     col("UnitCost").cast("double"))
    .withColumn("UnitPrice",    col("UnitPrice").cast("double"))
    .withColumn("UnitsInStock", col("UnitsInStock").cast("integer"))
    .withColumn("UnitsOnOrder", col("UnitsOnOrder").cast("integer"))
    .withColumn("ProductName",  trim(col("ProductName")))
    # Margin on current catalog pricing (not historical transaction pricing)
    .withColumn("MarginAbs",    round(col("UnitPrice") - col("UnitCost"), 2))
    .withColumn("MarginPct",
        when(col("UnitPrice") == 0, lit(0.0))
        .otherwise(round((col("UnitPrice") - col("UnitCost")) / col("UnitPrice") * 100, 1)))
    .withColumn("MarginBand",
        when(col("MarginPct") < 15, lit("Low"))
        .when(col("MarginPct") < 20, lit("Medium"))
        .otherwise(lit("High")))
    .withColumn("PriceTier",
        when(col("UnitPrice") < 10,  lit("Budget"))
        .when(col("UnitPrice") < 30, lit("Mid-Range"))
        .when(col("UnitPrice") < 60, lit("Premium"))
        .otherwise(lit("Luxury")))
    .withColumn("InventoryStatus",
        when(col("UnitsInStock") == 0,                     lit("Out of Stock"))
        .when(col("UnitsInStock") <= col("UnitsOnOrder"),  lit("Low Stock"))
        .otherwise(lit("In Stock")))
    .withColumn("_silver_at", current_timestamp())
    .dropDuplicates(["ProductID"])
)

print(f"silver_products: {df_silver.count()} rows")
df_silver.groupBy("MarginBand").count().show()
df_silver.groupBy("PriceTier").count().show()
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_products")


In [ ]:
# silver_employees and silver_offices
# v2 FIX: reads from Bronze Delta tables instead of re-reading raw Excel files.
# v2 FIX: Unknown Employee record (-1) injected for 5 orders with NULL EmployeeID.
# v2 FIX: Extension field handles both "-" and "nan" string values.

from datetime import datetime

# Read from Bronze (all columns are strings from astype(str) ingestion)
df_emp_raw = spark.table("bronze_employees")
df_off_raw = spark.table("bronze_offices")

df_emp_silver = (df_emp_raw
    .withColumnRenamed("EmpID", "EmployeeID")
    .withColumn("EmployeeID", col("EmployeeID").cast("integer"))
    .withColumn("Hire_Date", to_date(col("Hire_Date"), "yyyy-MM-dd"))
    .withColumn("Salary_Confidential",
        when(lower(col("Year_Salary")) == "confidential", lit(True)).otherwise(lit(False)))
    .withColumn("Year_Salary",
        when(col("Year_Salary").cast("double").isNotNull(), col("Year_Salary").cast("double"))
        .otherwise(lit(None).cast("double")))
    # v2 FIX: Dash or "nan" string means no phone extension
    .withColumn("Extension",
        when(col("Extension").isin("-", "nan"), lit(None)).otherwise(col("Extension")))
    .withColumn("Office", regexp_replace(trim(col("Office")), r"\.0$", ""))
    .withColumn("Reports_To",
        when(col("Reports_To").isNull() | (trim(col("Reports_To")).isin("", "nan")), lit(None).cast("integer"))
        .otherwise(col("Reports_To").cast("integer")))
    .withColumn("_silver_at", current_timestamp())
)

# v2 FIX: Inject Unknown Employee (ID = -1)
now = datetime.now()
unknown_emp = spark.createDataFrame(
    [(-1, "Unknown", "Employee", "Unknown", None, None, None, None, None, None, now)],
    ["EmployeeID", "Last_Name", "First_Name", "Title", "Hire_Date",
     "Office", "Extension", "Reports_To", "Year_Salary", "Salary_Confidential", "_silver_at"]
)
df_emp_silver = df_emp_silver.unionByName(unknown_emp)

df_off_silver = (df_off_raw
    .withColumn("OfficeStateProvince",
        when(col("OfficeStateProvince").isNull() | col("OfficeStateProvince").isin("nan",""), lit("N/A"))
        .otherwise(trim(col("OfficeStateProvince"))))
    .join(df_geo, trim(upper(col("OfficeCountry"))) == col("Country_Upper"), "left")
    .drop("Country_Upper", "DivisionID")
    .withColumn("Continent",  coalesce(col("Continent"), lit("Unknown")))
    .withColumn("SubRegion",  coalesce(col("SubRegion"),  lit("Unknown")))
    .withColumn("ISO2",       coalesce(col("ISO2"),       lit("??")))
    .withColumn("_silver_at", current_timestamp())
)

df_emp_silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_employees")
df_off_silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_offices")

print(f"silver_employees: {df_emp_silver.count()} rows (including Unknown Employee -1)")
print(f"silver_offices:   {df_off_silver.count()} rows")
df_emp_silver.filter(col("EmployeeID") == -1).show()


In [ ]:
# silver_budget
# v2 FIX: reads from Bronze Delta table instead of re-reading raw Excel.
# Forward-fill resolved using PySpark window function instead of pandas ffill.

from pyspark.sql import Window

df_bud_raw = spark.table("bronze_budget")

# Forward-fill Office: "nan" strings inherit last real value
w_ffill = Window.orderBy(monotonically_increasing_id()).rowsBetween(Window.unboundedPreceding, 0)

df_filled = (df_bud_raw
    .withColumn("Office",
        when(col("Office") == "nan", lit(None)).otherwise(
            regexp_replace(trim(col("Office")), r"\.0$", "")))
    .withColumn("Office", last(col("Office"), ignorenulls=True).over(w_ffill))
    .withColumn("EmployeeID", col("EmployeeID").cast("integer"))
)

year_cols = [c for c in df_filled.columns if c not in ("Office", "EmployeeID", "_ingested_at")]
stack_parts = [f"'{y}', `{y}`" for y in year_cols]
year_expr = ", ".join(stack_parts)

df_long = (df_filled
    .select("Office", "EmployeeID",
            expr(f"stack({len(year_cols)}, {year_expr}) as (BudgetYear, BudgetAmount)"))
    .withColumn("BudgetYear",   col("BudgetYear").cast("integer"))
    .withColumn("BudgetAmount",
        when(col("BudgetAmount") == "nan", lit(0.0))
        .otherwise(col("BudgetAmount").cast("double")))
    .withColumn("BudgetAmount", coalesce(col("BudgetAmount"), lit(0.0)))
    .withColumn("EmployeeID",   col("EmployeeID").cast("integer"))
    .withColumn("HasBudget",    col("BudgetAmount") > 0)
    .withColumn("BudgetMonthly", round(col("BudgetAmount") / 12, 2))
    .withColumn("_silver_at",   current_timestamp())
)

df_long.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("silver_budget")
print(f"silver_budget (long format): {df_long.count()} rows")
df_long.show(10)


In [ ]:
# Final Silver validation
print("=" * 60)
print("SILVER LAYER — VALIDATION SUMMARY")
print("=" * 60)

silver_tables = [t.tableName for t in spark.sql("SHOW TABLES").filter("tableName LIKE 'silver_%'").collect()]
for t in sorted(silver_tables):
    df = spark.table(t)
    nulls = [(c, df.filter(col(c).isNull()).count()) for c in df.columns if df.filter(col(c).isNull()).count() > 0]
    print(f"\n{t}: {df.count()} rows | {len(df.columns)} cols")
    for c, n in nulls:
        print(f"  NULL {c}: {n}")

print("\n--- Quarantine ---")
try:
    q = spark.table("bronze_quarantine")
    q.groupBy("_quarantine_check","_quarantine_source").count().show()
except:
    print("No records in quarantine")
